# LeetCode 897 — Increasing Order Search Tree bằng Python

**Link đề:** https://leetcode.com/problems/increasing-order-search-tree/
**Độ khó:** Easy · **Chủ đề:** Stack, Tree, Depth-First Search, Binary Search Tree, Binary Tree

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def increasingBST(self, root: TreeNode | None) -> TreeNode | None:
        ...
```

## Mục tiêu học
1. Hiểu **cây nhị phân tìm kiếm** (BST) và vì sao duyệt **in-order** (trái → gốc → phải) cho dãy **tăng dần**.
2. Viết in-order bằng **đệ quy** và bằng **stack**.
3. Đi từ cách "chép giá trị ra list rồi dựng cây mới" → "gom node rồi nối lại" → **nối lại tại chỗ trong lúc duyệt** với **node giả (dummy)** và con trỏ `cur`.
4. Trace từng bước việc nối `cur.right = node`, `node.left = None`; chứng minh invariant; phân tích Time/Space.
5. Kiểm thử: kết quả phải là "chuỗi bên phải" tăng dần, **không còn con trái nào**, dùng lại đúng các node cũ.

## 1. Đề bài và ví dụ

Cho `root` của một **BST**, hãy sắp xếp lại cây theo **thứ tự in-order** sao cho:
- node **nhỏ nhất** (trái nhất) trở thành **gốc mới**,
- mọi node **không có con trái**, và chỉ có **một con phải**.

Nói cách khác: biến cây thành một "dây xích" đi sang phải, giá trị tăng dần.

Ví dụ 1:

```text
Input:  [5,3,6,2,4,null,8,1,null,null,null,7,9]

              5
            /   \
           3     6
          / \     \
         2   4     8
        /         / \
       1         7   9

Output: [1,null,2,null,3,null,4,null,5,null,6,null,7,null,8,null,9]

   1
    \
     2
      \
       3  ... \
               9
```

Ví dụ 2: `[5,1,7]` → `[1,null,5,null,7]`.

**Constraints:** số node trong `[1, 100]`; `0 <= Node.val <= 1000`.

## 2. Nền tảng: `TreeNode`, dựng cây và in cây theo kiểu LeetCode

- `build_tree(values)`: dựng cây từ mảng level-order (dùng `None` cho `null`), dùng `deque` làm hàng đợi.
- `to_level_list(root)`: in cây ngược lại thành mảng level-order, bỏ `None` thừa ở cuối — đúng định dạng Output của LeetCode.

In [1]:
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left: "Optional[TreeNode]" = None, right: "Optional[TreeNode]" = None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); queue.append(node.right)
        i += 1
    return root

def to_level_list(root: Optional[TreeNode]) -> list:
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

EX1 = [5, 3, 6, 2, 4, None, 8, 1, None, None, None, 7, 9]
print(to_level_list(build_tree(EX1)))

[5, 3, 6, 2, 4, None, 8, 1, None, None, None, 7, 9]


### 2.1. BST là gì?

**Binary Search Tree**: với **mọi** node, mọi giá trị ở cây con trái **nhỏ hơn** node, mọi giá trị ở cây con phải **lớn hơn** node.

Hệ quả quan trọng: duyệt **in-order** (trái → gốc → phải) cho dãy giá trị **tăng dần**.

In [2]:
def inorder_values(node: Optional[TreeNode]) -> list[int]:
    if node is None:
        return []
    return inorder_values(node.left) + [node.val] + inorder_values(node.right)

print("In-order của ví dụ 1:", inorder_values(build_tree(EX1)))

In-order của ví dụ 1: [1, 2, 3, 4, 5, 6, 7, 8, 9]


### 2.2. Ba kiểu duyệt DFS — chỉ khác **vị trí** của "thăm gốc"

| Kiểu | Thứ tự | Ví dụ với cây `2 (1, 3)` |
|---|---|---|
| Pre-order | gốc → trái → phải | 2, 1, 3 |
| **In-order** | trái → **gốc** → phải | **1, 2, 3** |
| Post-order | trái → phải → gốc | 1, 3, 2 |

In [3]:
def preorder(n):  return [] if n is None else [n.val] + preorder(n.left) + preorder(n.right)
def inorder(n):   return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)
def postorder(n): return [] if n is None else postorder(n.left) + postorder(n.right) + [n.val]

t = build_tree([2, 1, 3])
print("pre :", preorder(t)); print("in  :", inorder(t)); print("post:", postorder(t))

pre : [2, 1, 3]
in  : [1, 2, 3]
post: [1, 3, 2]


### 2.3. Hàm kiểm tra kết quả: "dây xích phải" hợp lệ

Để test, ta viết `is_right_chain(root)`: mọi node không có con trái; và `chain_values(root)` đi theo `.right` gom giá trị.

In [4]:
def chain_values(root: Optional[TreeNode]) -> list[int]:
    out = []
    while root:
        out.append(root.val)
        root = root.right
    return out

def is_right_chain(root: Optional[TreeNode]) -> bool:
    while root:
        if root.left is not None:
            return False
        root = root.right
    return True

chain = TreeNode(1, None, TreeNode(5, None, TreeNode(7)))
print(chain_values(chain), is_right_chain(chain), to_level_list(chain))

[1, 5, 7] True [1, None, 5, None, 7]


## 3. Cách 1 — Chép giá trị ra list rồi dựng cây mới

Ý tưởng tự nhiên:
1. Lấy in-order → list giá trị tăng dần.
2. Dựng một dây xích **node mới** từ list đó.

Đúng kết quả, nhưng tạo `n` node mới (bộ nhớ `O(n)` cho list + `O(n)` node mới) và bỏ đi các node cũ.

In [5]:
def increasing_bst_rebuild(root: Optional[TreeNode]) -> Optional[TreeNode]:
    vals = inorder_values(root)
    dummy = TreeNode(-1)                 # node giả đứng trước gốc mới
    cur = dummy
    for v in vals:
        cur.right = TreeNode(v)
        cur = cur.right
    return dummy.right

print(to_level_list(increasing_bst_rebuild(build_tree(EX1))))
print(to_level_list(increasing_bst_rebuild(build_tree([5, 1, 7]))))

[1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7, None, 8, None, 9]
[1, None, 5, None, 7]


### 3.1. Kỹ thuật **node giả (dummy / sentinel)**

Khi dựng dây xích, node đầu tiên là trường hợp đặc biệt ("chưa có node trước nó để gắn vào"). Mẹo: tạo một node **giả** `dummy`, coi nó là "node trước" của node đầu. Sau đó mọi node đều được xử lý **giống nhau**: `cur.right = mới; cur = mới`. Cuối cùng trả `dummy.right`.

In [6]:
# Không dùng dummy: phải xử lý riêng node đầu
def build_chain_no_dummy(vals):
    head = None
    cur = None
    for v in vals:
        node = TreeNode(v)
        if head is None:            # trường hợp đặc biệt
            head = cur = node
        else:
            cur.right = node
            cur = node
    return head

print(chain_values(build_chain_no_dummy([1, 2, 3])), "<- đúng nhưng nhiều nhánh if hơn")

[1, 2, 3] <- đúng nhưng nhiều nhánh if hơn


## 4. Cách 2 — Gom chính các **node** theo in-order rồi nối lại

Đề muốn "sắp xếp lại cây" → tốt hơn là **dùng lại** node cũ. Gom node (không phải giá trị) vào list, rồi với mỗi node: `left = None`, `right = node kế tiếp`.

In [7]:
def increasing_bst_nodes(root: Optional[TreeNode]) -> Optional[TreeNode]:
    nodes = []
    def inorder_nodes(node):
        if node is None:
            return
        inorder_nodes(node.left)
        nodes.append(node)
        inorder_nodes(node.right)
    inorder_nodes(root)

    for i in range(len(nodes)):
        nodes[i].left = None
        nodes[i].right = nodes[i + 1] if i + 1 < len(nodes) else None
    return nodes[0] if nodes else None

print(to_level_list(increasing_bst_nodes(build_tree(EX1))))

[1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7, None, 8, None, 9]


**Nút thắt:** vẫn cần list `nodes` kích thước `O(n)`. Câu hỏi: *có thể nối ngay **trong lúc** duyệt in-order, không cần list không?*

**Insight:** in-order thăm node theo đúng thứ tự cần nối. Chỉ cần nhớ **node vừa nối gần nhất** (`cur`). Khi thăm node mới: `cur.right = node`, `node.left = None`, `cur = node`. Kết hợp dummy để khỏi xử lý riêng node đầu.

## 5. Cách 3 — Lời giải chính: nối lại tại chỗ khi duyệt in-order (đệ quy)

Mảnh ghép:
1. `dummy = TreeNode(0)`, `self.cur = dummy` (dùng thuộc tính để hàm đệ quy thay đổi được).
2. `inorder(node)`:
   - `inorder(node.left)`
   - **thăm node**: `node.left = None`; `self.cur.right = node`; `self.cur = node`
   - `inorder(node.right)`
3. Trả `dummy.right`.

**An toàn không?** Khi thăm `node`, ta đặt `node.left = None` — nhưng cây con trái của `node` **đã được duyệt xong** trước đó, nên không mất gì. `node.right` không bị sửa ở bước này nên lời gọi `inorder(node.right)` ngay sau vẫn đi đúng. Còn `cur.right = node` ghi đè con phải của node **liền trước** — mục 9 giải thích vì sao việc này không làm mất node nào chưa duyệt.

In [8]:
def increasing_bst_inplace(root: Optional[TreeNode]) -> Optional[TreeNode]:
    dummy = TreeNode(0)
    cur = dummy

    def inorder(node):
        nonlocal cur                     # cho phép gán lại biến cur của hàm ngoài
        if node is None:
            return
        inorder(node.left)
        node.left = None                 # cắt con trái (đã duyệt xong)
        cur.right = node                 # nối vào đuôi dây xích
        cur = node                       # node thành đuôi mới
        inorder(node.right)

    inorder(root)
    return dummy.right

res = increasing_bst_inplace(build_tree(EX1))
print(to_level_list(res))
print("chain:", chain_values(res), "| không còn con trái:", is_right_chain(res))

[1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7, None, 8, None, 9]
chain: [1, 2, 3, 4, 5, 6, 7, 8, 9] | không còn con trái: True


### 5.1. `nonlocal` — khi nào cần?

Hàm lồng nhau **đọc** được biến của hàm ngoài, nhưng nếu **gán** `cur = node` thì Python coi `cur` là biến **cục bộ mới** của hàm trong → lỗi `UnboundLocalError` khi đọc `cur.right` trước đó. `nonlocal cur` báo "cur là biến của hàm bao ngoài".

(Ở cách 1 của bài 872 ta chỉ gọi `out.append`, không gán lại `out`, nên không cần `nonlocal`.)

In [9]:
def outer_bad():
    count = 0
    def inc():
        count += 1          # gán -> Python coi count là biến cục bộ của inc
    try:
        inc()
    except UnboundLocalError as e:
        print("Không có nonlocal:", e)

def outer_good():
    count = 0
    def inc():
        nonlocal count
        count += 1
    inc(); inc()
    print("Có nonlocal: count =", count)

outer_bad(); outer_good()

Không có nonlocal: cannot access local variable 'count' where it is not associated with a value
Có nonlocal: count = 2


## 6. Cách 4 — In-order bằng stack (không đệ quy), nối tại chỗ

Mẫu in-order lặp kinh điển:
1. Đi xuống **trái hết mức**, đẩy từng node vào stack.
2. `pop` một node → **thăm** nó (ở đây: nối vào dây xích).
3. Chuyển sang `node.right` và lặp lại.

Phải lưu `nxt = node.right` **trước** khi nối? Ở đây ta không sửa `node.right` khi thăm (chỉ sửa `cur.right` của node trước), nên đọc `node.right` sau khi thăm vẫn đúng. Nhưng việc `cur.right = node` có thể ghi đè `cur.right` cũ — ta chứng minh ở mục 9 rằng lúc đó cây con phải của `cur` đã duyệt xong.

In [10]:
def increasing_bst_stack(root: Optional[TreeNode]) -> Optional[TreeNode]:
    dummy = TreeNode(0)
    cur = dummy
    stack = []
    node = root
    while stack or node:
        while node:                      # 1) xuống trái hết mức
            stack.append(node)
            node = node.left
        node = stack.pop()               # 2) thăm
        node.left = None
        cur.right = node
        cur = node
        node = node.right                # 3) sang phải
    return dummy.right

print(to_level_list(increasing_bst_stack(build_tree(EX1))))
print(to_level_list(increasing_bst_stack(build_tree([5, 1, 7]))))

[1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7, None, 8, None, 9]
[1, None, 5, None, 7]


## 7. TRACE từng bước

### 7.1. Bảng trace cách 3 với `[5,1,7]` (cây `5` có con trái `1`, con phải `7`)

| Bước | Node thăm | Trước: cur | Thao tác | Dây xích sau (từ dummy) |
|---:|---:|---|---|---|
| 1 | 1 | dummy | `1.left=None; dummy.right=1; cur=1` | D → 1 |
| 2 | 5 | 1 | `5.left=None` (cắt 1); `1.right=5; cur=5` | D → 1 → 5 |
| 3 | 7 | 5 | `7.left=None`; `5.right=7` (vốn đã là 7); `cur=7` | D → 1 → 5 → 7 |

In [11]:
def trace_increasing_bst(values):
    root = build_tree(values)
    dummy = TreeNode("D")
    cur = dummy
    step = 0

    def inorder(node, depth=0):
        nonlocal cur, step
        if node is None:
            return
        inorder(node.left, depth + 1)
        step += 1
        old_left = node.left.val if node.left else None
        print(f"Bước {step}: thăm {node.val} (độ sâu {depth}) | cắt left={old_left} | nối {cur.val}.right = {node.val}")
        node.left = None
        cur.right = node
        cur = node
        inorder(node.right, depth + 1)

    inorder(root)
    print("Dây xích:", chain_values(dummy.right))
    return dummy.right

_ = trace_increasing_bst([5, 1, 7]); print()
_ = trace_increasing_bst(EX1)

Bước 1: thăm 1 (độ sâu 1) | cắt left=None | nối D.right = 1
Bước 2: thăm 5 (độ sâu 0) | cắt left=1 | nối 1.right = 5
Bước 3: thăm 7 (độ sâu 1) | cắt left=None | nối 5.right = 7
Dây xích: [1, 5, 7]

Bước 1: thăm 1 (độ sâu 3) | cắt left=None | nối D.right = 1
Bước 2: thăm 2 (độ sâu 2) | cắt left=1 | nối 1.right = 2
Bước 3: thăm 3 (độ sâu 1) | cắt left=2 | nối 2.right = 3
Bước 4: thăm 4 (độ sâu 2) | cắt left=None | nối 3.right = 4
Bước 5: thăm 5 (độ sâu 0) | cắt left=3 | nối 4.right = 5
Bước 6: thăm 6 (độ sâu 1) | cắt left=None | nối 5.right = 6
Bước 7: thăm 7 (độ sâu 3) | cắt left=None | nối 6.right = 7
Bước 8: thăm 8 (độ sâu 2) | cắt left=7 | nối 7.right = 8
Bước 9: thăm 9 (độ sâu 3) | cắt left=None | nối 8.right = 9
Dây xích: [1, 2, 3, 4, 5, 6, 7, 8, 9]


### 7.2. Trace cách stack — nhìn stack thay đổi

In [12]:
def trace_stack(values):
    root = build_tree(values)
    dummy = TreeNode("D"); cur = dummy
    stack, node = [], root
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        print(f"  stack sau khi xuống trái: {[n.val for n in stack]}")
        node = stack.pop()
        node.left = None
        cur.right = node
        print(f"  pop {node.val} -> nối {cur.val}.right = {node.val}")
        cur = node
        node = node.right
    print("Dây xích:", chain_values(dummy.right))

trace_stack([5, 3, 6, 2, 4])

  stack sau khi xuống trái: [5, 3, 2]
  pop 2 -> nối D.right = 2
  stack sau khi xuống trái: [5, 3]
  pop 3 -> nối 2.right = 3
  stack sau khi xuống trái: [5, 4]
  pop 4 -> nối 3.right = 4
  stack sau khi xuống trái: [5]
  pop 5 -> nối 4.right = 5
  stack sau khi xuống trái: [6]
  pop 6 -> nối 5.right = 6
Dây xích: [2, 3, 4, 5, 6]


## 8. Code nộp LeetCode

In [13]:
class Solution:
    def increasingBST(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        dummy = TreeNode(0)
        self.cur = dummy

        def inorder(node):
            if node is None:
                return
            inorder(node.left)
            node.left = None
            self.cur.right = node
            self.cur = node
            inorder(node.right)

        inorder(root)
        return dummy.right

sol = Solution()
print(to_level_list(sol.increasingBST(build_tree(EX1))))
print(to_level_list(sol.increasingBST(build_tree([5, 1, 7]))))

[1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7, None, 8, None, 9]
[1, None, 5, None, 7]


Ở bản nộp, ta dùng `self.cur` (thuộc tính của đối tượng) thay cho `nonlocal` — cả hai đều được; `self.cur` quen thuộc với người viết Java/C++.

### 8.1. Bản nộp dùng stack (tránh giới hạn đệ quy với cây rất lệch)

In [14]:
class SolutionStack:
    def increasingBST(self, root: Optional[TreeNode]) -> Optional[TreeNode]:
        dummy = cur = TreeNode(0)
        stack, node = [], root
        while stack or node:
            while node:
                stack.append(node)
                node = node.left
            node = stack.pop()
            node.left = None
            cur.right = node
            cur = node
            node = node.right
        return dummy.right

print(to_level_list(SolutionStack().increasingBST(build_tree(EX1))))

[1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7, None, 8, None, 9]


## 9. Vì sao đúng?

**Thứ tự đúng:** BST + in-order ⇒ node được thăm theo giá trị tăng dần (mục 2.1). Mỗi node được nối vào **đuôi** dây xích đúng lúc được thăm ⇒ dây xích tăng dần, chứa mọi node đúng một lần.

**Invariant:** ngay trước khi thăm một node `x`:
- `dummy.right → ... → cur` là dây xích (không con trái) gồm **đúng** các node đã thăm, theo thứ tự in-order;
- mọi node chưa thăm vẫn còn **nguyên liên kết gốc** có thể tới được từ stack/lời gọi đệ quy đang chờ.

**Không phá phần chưa duyệt:**
- `x.left = None`: cây con trái của `x` đã duyệt xong (in-order thăm trái trước) ⇒ an toàn.
- `cur.right = x`: `cur` là node **liền trước** `x` trong in-order. Có hai khả năng: (a) `x` nằm trong cây con phải của `cur` — khi đó `x` là node trái nhất của cây con phải ấy, và ta đã đi qua `cur.right` để xuống tới `x` (đệ quy đã lưu đường đi), nên ghi đè `cur.right` không làm mất node nào chưa thăm; (b) `cur` **không có** con phải còn chưa thăm (cây con phải rỗng hoặc đã duyệt xong) ⇒ ghi đè an toàn.

**Kết thúc:** node cuối cùng được thăm là node lớn nhất; `right` của nó vốn là `None` (nó không có con phải, nếu không đã có node lớn hơn) ⇒ dây xích kết thúc đúng.

## 10. Độ phức tạp và so sánh

`n` = số node, `h` = chiều cao cây (`log n` nếu cân bằng, `n` nếu lệch).

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | Dùng lại node? |
|---|---|---|---|:---:|
| 1. Giá trị → cây mới | in-order list + dựng mới | `O(n)` | `O(n)` list + `O(n)` node mới | không |
| 2. Gom node rồi nối | list node | `O(n)` | `O(n)` | có |
| 3. Nối khi duyệt (đệ quy) | dummy + cur | `O(n)` | `O(h)` ngăn xếp đệ quy | **có** |
| 4. Nối khi duyệt (stack) | in-order lặp | `O(n)` | `O(h)` stack | **có** |

Lưu ý: `inorder_values` ở cách 1 dùng nối list `+` ở mỗi lời gọi nên thực tế có thể tốn `O(n·h)`; muốn chặt `O(n)` thì dùng `append` vào list chung (như cách 2).

(Nâng cao: **Morris traversal** đạt `O(1)` bộ nhớ thêm bằng cách tạm "luồn" con trỏ — ngoài phạm vi bài Easy.)

## 11. Bộ test: mẫu đề, biên, đối chiếu

In [15]:
def run(f, values):
    root = build_tree(values)
    original_nodes = set()
    def collect(n):
        if n:
            original_nodes.add(id(n)); collect(n.left); collect(n.right)
    collect(root)
    res = f(root)
    assert is_right_chain(res), "còn con trái!"
    # kiểm tra dùng lại node cũ (trừ cách 1 tạo node mới)
    reused = all(id(n) in original_nodes for n in iter_chain(res))
    return chain_values(res), to_level_list(res), reused

def iter_chain(root):
    while root:
        yield root
        root = root.right

methods = {
    "rebuild": increasing_bst_rebuild,
    "nodes": increasing_bst_nodes,
    "inplace": increasing_bst_inplace,
    "stack": increasing_bst_stack,
    "Solution": Solution().increasingBST,
    "SolutionStack": SolutionStack().increasingBST,
}

cases = [
    ("đề VD1", EX1, [1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7, None, 8, None, 9]),
    ("đề VD2", [5, 1, 7], [1, None, 5, None, 7]),
    ("1 node", [42], [42]),
    ("đã là dây phải", [1, None, 2, None, 3], [1, None, 2, None, 3]),
    ("lệch trái hoàn toàn", [3, 2, None, 1], [1, None, 2, None, 3]),
    ("gốc là min", [2, None, 4, 3], [2, None, 3, None, 4]),
]
for name, values, expected in cases:
    for mname, f in methods.items():
        vals, level, reused = run(f, values)
        assert level == expected, (name, mname, level)
        assert reused == (mname != "rebuild"), (name, mname)
    print(f"PASS  {name}: {expected}")
print("Mọi cách in-place đều dùng lại node cũ; cách rebuild tạo node mới (đúng như phân tích).")

PASS  đề VD1: [1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7, None, 8, None, 9]
PASS  đề VD2: [1, None, 5, None, 7]
PASS  1 node: [42]
PASS  đã là dây phải: [1, None, 2, None, 3]
PASS  lệch trái hoàn toàn: [1, None, 2, None, 3]
PASS  gốc là min: [2, None, 3, None, 4]
Mọi cách in-place đều dùng lại node cũ; cách rebuild tạo node mới (đúng như phân tích).


`id(obj)` trả về "danh tính" của đối tượng trong bộ nhớ. Ta lưu `id` các node gốc để kiểm tra node trong kết quả có phải **chính** các node cũ không.

In [16]:
import random

def random_bst(values):
    root = None
    def insert(node, v):
        if node is None:
            return TreeNode(v)
        if v < node.val:
            node.left = insert(node.left, v)
        else:
            node.right = insert(node.right, v)
        return node
    for v in values:
        root = insert(root, v)
    return root

random.seed(897)
for _ in range(300):
    vals = random.sample(range(0, 1001), random.randint(1, 30))
    expected = sorted(vals)
    for f in methods.values():
        res = f(random_bst(vals))
        assert is_right_chain(res) and chain_values(res) == expected
print("300 BST ngẫu nhiên (giá trị khác nhau): mọi cách cho dây xích tăng dần đúng.")

300 BST ngẫu nhiên (giá trị khác nhau): mọi cách cho dây xích tăng dần đúng.


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên `node.left = None`
Dây xích phải đúng thứ tự, nhưng các node vẫn giữ con trái cũ → cây có **chu trình**/không phải dây xích → LeetCode báo sai (hoặc treo khi in).

In [17]:
def wrong_keep_left(root):
    dummy = TreeNode(0); cur = dummy
    def inorder(node):
        nonlocal cur
        if node is None:
            return
        inorder(node.left)
        cur.right = node                   # quên node.left = None
        cur = node
        inorder(node.right)
    inorder(root)
    return dummy.right

res = wrong_keep_left(build_tree([5, 1, 7]))
print("Thứ tự theo .right:", chain_values(res), "| là dây phải hợp lệ?", is_right_chain(res))
print("Node 5 vẫn còn left =", res.right.left.val, "-> trỏ ngược về 1, tạo vòng!")

Thứ tự theo .right: [1, 5, 7] | là dây phải hợp lệ? False
Node 5 vẫn còn left = 1 -> trỏ ngược về 1, tạo vòng!


### Lỗi 2: Trả `dummy` thay vì `dummy.right`
Kết quả sẽ có thêm node giả `0` ở đầu.

### Lỗi 3: Gán `cur = node` trong hàm lồng mà quên `nonlocal`/`self`
→ `UnboundLocalError` (xem mục 5.1).

### Lỗi 4: Dùng pre-order hoặc post-order
Thứ tự không tăng dần → sai.

In [18]:
def wrong_preorder(root):
    dummy = TreeNode(0); cur = dummy
    nodes = []
    def pre(node):
        if node:
            nodes.append(node); pre(node.left); pre(node.right)
    pre(root)
    for n in nodes:
        n.left = None; cur.right = n; cur = n
    cur.right = None
    return dummy.right

print("Pre-order (sai):", chain_values(wrong_preorder(build_tree(EX1))))
print("In-order (đúng):", chain_values(Solution().increasingBST(build_tree(EX1))))
print("Trả dummy (sai):", chain_values(TreeNode(0, None, Solution().increasingBST(build_tree([5, 1, 7])))))

Pre-order (sai): [5, 3, 2, 1, 4, 6, 8, 7, 9]
In-order (đúng): [1, 2, 3, 4, 5, 6, 7, 8, 9]
Trả dummy (sai): [0, 1, 5, 7]


### Lỗi 5: Quên đặt `right = None` cho node cuối ở cách "gom node"
Ở cách 2, node lớn nhất phải có `right = None`. Nếu chỉ gán `nodes[i].right = nodes[i+1]` cho `i < n-1` thì node cuối vẫn giữ `right` cũ — thường là `None` (node lớn nhất không có con phải), nên lỗi này hay "ẩn". Nhưng viết tường minh vẫn an toàn hơn.

## 13. Mẹo nhận dạng pattern

- **BST** + "thứ tự tăng dần / sắp xếp / phần tử thứ k / trước-sau" → nghĩ ngay **in-order**.
- "Biến cây thành danh sách liên kết / dây xích" → in-order + **con trỏ `prev`/`cur`** + **dummy**.
- Muốn tránh đệ quy → mẫu in-order bằng stack: *xuống trái hết mức → pop → sang phải*.
- Sửa con trỏ trong lúc duyệt: luôn tự hỏi "phần nào **chưa duyệt** có bị mất liên kết không?".

Bài cùng họ: 94 Binary Tree Inorder Traversal, 114 Flatten Binary Tree to Linked List (pre-order), 230 Kth Smallest Element in a BST, 426 Convert BST to Sorted Doubly Linked List, 98 Validate BST.

## 14. Biến thể

### 14.1. Dây xích **giảm dần** (đi theo `right`)
In-order **ngược**: phải → gốc → trái.

### 14.2. Phần tử nhỏ thứ k (LeetCode 230)
Dừng in-order ở node thứ `k`.

In [19]:
def decreasing_chain(root):
    dummy = TreeNode(0); cur = dummy
    def rev_inorder(node):
        nonlocal cur
        if node is None:
            return
        rev_inorder(node.right)
        right_child = node.left            # lưu trước vì sắp sửa con trỏ
        node.left = None
        cur.right = node
        cur = node
        rev_inorder(right_child)
    rev_inorder(root)
    cur.right = None
    return dummy.right

def kth_smallest(root, k):
    stack, node = [], root
    while stack or node:
        while node:
            stack.append(node); node = node.left
        node = stack.pop()
        k -= 1
        if k == 0:
            return node.val
        node = node.right

print("Giảm dần:", chain_values(decreasing_chain(build_tree(EX1))))
print("k=3 nhỏ nhất:", kth_smallest(build_tree(EX1), 3))

Giảm dần: [9, 8, 7, 6, 5, 4, 3, 2, 1]
k=3 nhỏ nhất: 3


Ở `decreasing_chain` có một chỗ tinh tế: ta đi phải trước, rồi khi thăm `node` ta **ghi đè `cur.right`** — mà `cur` là node lớn hơn đã thăm, nên an toàn; nhưng `node.left` (cây con trái, **chưa duyệt**) phải được **lưu lại** (`right_child = node.left`) trước khi đặt `node.left = None`. Đây đúng là câu hỏi "phần nào chưa duyệt có bị mất liên kết không?".

## 15. Bài tập tự luyện

Hàm TODO chỉ định nghĩa, không được gọi khi chạy notebook.

### Bài 1 — Tính tay
Cho BST `[4, 2, 6, 1, 3, 5, 7]`. Viết thứ tự in-order và vẽ dây xích kết quả. Viết luôn Output theo định dạng LeetCode.

### Bài 2 — Dự đoán output
Với cây `[2, 1, 3]`, `preorder`, `inorder`, `postorder` trả về gì? Và `to_level_list(TreeNode(1, None, TreeNode(2)))`?

### Bài 3 — In-order lặp bằng stack (chỉ trả list giá trị)

In [20]:
def inorder_iter(root) -> list[int]:
    # TODO: xuống trái hết mức -> pop -> sang phải
    pass

# assert inorder_iter(build_tree(EX1)) == [1, 2, 3, 4, 5, 6, 7, 8, 9]

### Bài 4 — Tự viết `increasingBST` (đệ quy + dummy + nonlocal)

In [21]:
def increasing_bst_exercise(root):
    # TODO
    pass

# res = increasing_bst_exercise(build_tree([5, 1, 7]))
# assert to_level_list(res) == [1, None, 5, None, 7]

### Bài 5 — Biến thể: kiểm tra một cây có phải BST hợp lệ (in-order tăng **nghiêm ngặt**)

In [22]:
def is_valid_bst(root) -> bool:
    # TODO: in-order, so với giá trị trước đó
    pass

# assert is_valid_bst(build_tree([2, 1, 3])) is True
# assert is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6])) is False

### Bài 6 — Mini challenge: Flatten theo **pre-order** (LeetCode 114, ý tưởng)
Biến cây (không nhất thiết BST) thành dây xích phải theo thứ tự **pre-order**, dùng lại node.

In [23]:
def flatten_preorder(root):
    # TODO: gom node theo pre-order rồi nối, hoặc duyệt phải-trái-gốc với prev
    pass

# r = build_tree([1, 2, 5, 3, 4, None, 6]); flatten_preorder(r)
# assert chain_values(r) == [1, 2, 3, 4, 5, 6] and is_right_chain(r)

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | BST → in-order tăng | 1..7 | `[1,null,2,null,...,null,7]` |
| 2 | vị trí của gốc | in-order giữa | `[2,1,3]`, `[1,2,3]`, `[1,3,2]`, `[1,None,2]` |
| 3 | stack | `while node: push; node = node.left` | `node = stack.pop(); out.append; node = node.right` |
| 4 | dummy + cur | `nonlocal cur` | cắt left, nối right, cập nhật cur |
| 5 | in-order tăng nghiêm ngặt | nhớ `prev` | `if prev is not None and node.val <= prev: False` |
| 6 | pre-order | gom list node | `n.left = None; n.right = next` |

## 17. Đáp án tham khảo

**Bài 1:** in-order `1,2,3,4,5,6,7` → Output `[1,null,2,null,3,null,4,null,5,null,6,null,7]`.

**Bài 2:** pre `[2,1,3]`, in `[1,2,3]`, post `[1,3,2]`; `to_level_list(...) = [1, None, 2]`.

In [24]:
def ans_inorder_iter(root):
    out, stack, node = [], [], root
    while stack or node:
        while node:
            stack.append(node); node = node.left
        node = stack.pop()
        out.append(node.val)
        node = node.right
    return out

def ans_increasing_bst(root):
    dummy = TreeNode(0); cur = dummy
    def inorder(node):
        nonlocal cur
        if node is None:
            return
        inorder(node.left)
        node.left = None
        cur.right = node
        cur = node
        inorder(node.right)
    inorder(root)
    return dummy.right

def ans_is_valid_bst(root):
    prev = None
    for v in ans_inorder_iter(root):
        if prev is not None and v <= prev:
            return False
        prev = v
    return True

def ans_flatten_preorder(root):
    nodes = []
    def pre(n):
        if n:
            nodes.append(n); pre(n.left); pre(n.right)
    pre(root)
    for i, n in enumerate(nodes):
        n.left = None
        n.right = nodes[i + 1] if i + 1 < len(nodes) else None

assert to_level_list(Solution().increasingBST(build_tree([4, 2, 6, 1, 3, 5, 7]))) == [1, None, 2, None, 3, None, 4, None, 5, None, 6, None, 7]
assert (preorder(build_tree([2, 1, 3])), inorder(build_tree([2, 1, 3])), postorder(build_tree([2, 1, 3]))) == ([2, 1, 3], [1, 2, 3], [1, 3, 2])
assert to_level_list(TreeNode(1, None, TreeNode(2))) == [1, None, 2]
assert ans_inorder_iter(build_tree(EX1)) == [1, 2, 3, 4, 5, 6, 7, 8, 9]
assert to_level_list(ans_increasing_bst(build_tree([5, 1, 7]))) == [1, None, 5, None, 7]
assert ans_is_valid_bst(build_tree([2, 1, 3])) is True
assert ans_is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6])) is False
r = build_tree([1, 2, 5, 3, 4, None, 6]); ans_flatten_preorder(r)
assert chain_values(r) == [1, 2, 3, 4, 5, 6] and is_right_chain(r)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def increasingBST(self, root):
        dummy = TreeNode(0)
        self.cur = dummy                      # đuôi của dây xích

        def inorder(node):                    # trái -> gốc -> phải
            if node is None:
                return
            inorder(node.left)
            node.left = None                  # cắt trái (đã duyệt xong)
            self.cur.right = node             # nối vào đuôi
            self.cur = node                   # cập nhật đuôi
            inorder(node.right)

        inorder(root)
        return dummy.right                    # bỏ node giả
```

**Câu nhớ:** *"BST + in-order = tăng dần; dummy + cur để nối đuôi."*

**Checklist:**
- [ ] Duyệt **in-order** (không phải pre/post)?
- [ ] `node.left = None` cho **mọi** node?
- [ ] Trả `dummy.right`, không phải `dummy`?
- [ ] `nonlocal` hoặc `self.cur` khi gán lại `cur` trong hàm lồng?
- [ ] Test 1 node, cây lệch trái, cây đã là dây phải?